# Trio (SINGLE-insertion, `--ins 0`): loc1 vs disp1 vs exact1 ($G_t$, $G_s$), lattice prop, $L=1,2,4$

All three are a SINGLE insertion with a RAW trace $\mathrm{tr}[W(t_0)\,P\,W(t)\,P]$ (no sum over
sites/links, no summation weight) -- apples-to-apples.

- **loc1** (`jj_local_deter --ins 0`, `corr_deter_local1_L*`): on-site bare $\sigma^a$ at SITE 0.
  $G_t=$`s3`, $G_s=$(`s1`+`s2`)$/(D{-}1)$ (averaged over the two transverse Pauli directions).
- **disp1** (`jj_disp_deter --ins 0`, `corr_deter_disp1_L*`): displaced Wilson point-split current.
  sp = SPATIAL link 0 ($W_{ov\kappa}$, a **single** direction $e^a\sigma_a$); tp = TEMPORAL time-link at SITE 0.
- **exact1** (`jj_exact_diag_deter_free`, `corr_deter_exact1_L*`): bare overlap conserved current $K/\kappa$,
  tp:SITE 0, sp:LINK 0 (single direction), plus the GW resolvent dressing.

**Own-operator ratio $G_s/G_t$** (qed3int_v2-13, Eq.4.28: $G^s\equiv\frac{\delta^{ab}-e_3^a e_3^b}{D-1}f^{ab}$ is the
**per-direction average**): so $G_s/G_t\to -1$ for all three (opposite sign, equal magnitude).  loc1 averages
its two Pauli channels ($/(D{-}1)$); disp1/exact1 are a single link (one direction) so no division.
exact1 vs disp1 at the same link isolates the GW resolvent dressing.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

dir1 = '/mnt/barracuda22/qed3/qed3/src/both_3d/'
ESN  = dir1 + 'data_free_vmRe0.000000vmIm0.000000/'
at_cft = 0.2
Nt_cft = 128
Delta_cft = 2.0
Dm1 = 2.0   # D - 1 : spatial-projector average (qed3int_v2-13 Eq.4.28: G^s = (delta-e3e3)/(D-1) f).
tag = ''    # '' = lattice prop ; 'cont_' = continuum prop

def cft_shape(t):
    return np.exp(-Delta_cft*t) * (1.0 - np.exp(-t))**(-2.0*Delta_cft)

def load_Vpp(corrdir, proj, origin='t0_0', k=0):
    path = ESN + corrdir + '/corr.' + str(k) + '.h5'
    with h5py.File(path, 'r') as f:
        key = 'h0/' + origin + '/' + proj + '/Vpp'
        return f[key+'/real'][()] + 1j*f[key+'/imag'][()]

# SINGLE-insertion loaders (--ins 0): loc1 = site 0, disp1 = link0(sp)/site0(tp), exact1 = site/link 0.
# G_s is the v2-13 per-direction (projector-averaged) spatial correlator:
#   loc1 sums s1+s2 (BOTH transverse dirs at the site) -> divide by Dm1 = D-1;
#   disp1/exact1 are a SINGLE link (one direction) -> already per-direction, NO division.
# So all three give G_s/G_t -> -1 (opposite sign to tp).
def loc1_Gt(L):  cd='corr_deter_local1_%sL%d'%(tag,L); return load_Vpp(cd,'s3').real
def loc1_Gs(L):  cd='corr_deter_local1_%sL%d'%(tag,L); return ((load_Vpp(cd,'s1')+load_Vpp(cd,'s2'))/Dm1).real
def disp1_Gs(L): return load_Vpp('corr_deter_disp1_%sL%d'%(tag,L),'sp').real
def disp1_Gt(L): return load_Vpp('corr_deter_disp1_%sL%d'%(tag,L),'tp').real
def ex_Gt(L):    return load_Vpp('corr_deter_exact1_%sL%d'%(tag,L),'tp').real
def ex_Gs(L):    return load_Vpp('corr_deter_exact1_%sL%d'%(tag,L),'sp').real

In [ ]:
# tp (G_t): normalize each estimator by c_e = cft_shape(t_ref)/Gt_e(t_ref) -- a SINGLE constant per
# estimator matching the lattice tp to the analytic (4.31) shape at the reference (ref=8 = your "t=8",
# i.e. dt index 8, t=1.6).  sign(c_e) IS the lattice Gt sign (printed below).  c_e*Gt_e then overlays the
# +cft_shape by construction; log axis is positive via the CFT match (no per-curve sign flips).
dt = np.arange(1, int(Nt_cft/2)); t = dt*at_cft; ref = 8
plt.figure()
print('%-6s %-15s %-15s %-15s   (sign(c) = lattice Gt sign)' % ('L','c_loc','c_disp','c_exact'))
for L in [1, 2, 4]:
    row = []
    for nm, f, mk in [('loc', loc1_Gt, 'o-'), ('disp', disp1_Gt, 'd-.'), ('exact', ex_Gt, 's--')]:
        try:
            g = f(L); c = cft_shape(t[ref-1]) / g[ref]
            plt.plot(t, c*g[dt], mk, ms=3, label='%s L=%d' % (nm, L))
            row.append('% .3e' % c)
        except (OSError, KeyError):
            row.append('   --        ')
    print('L=%-4d %-15s %-15s %-15s' % (L, *row))
plt.plot(t, cft_shape(t), 'k-', label='(4.31) shape')
plt.yscale('log'); plt.xlabel(r'$t = a_t n_t$'); plt.ylabel(r'$c\,G_t$ (matched to CFT at ref)')
plt.title('tp: lattice $G_t$ matched to (4.31) at dt=%d (sign in c)' % ref); plt.legend(fontsize=8)
plt.savefig('comp_trio_1_sum_tp.pdf', bbox_inches='tight')

In [ ]:
# sp (G_s): use the SAME c_e (from tp at ref) to normalize Gs -> c_e*Gs_e carries the PHYSICAL sign
# relative to Gt.  CFT predicts Gs = -Gt (ratio -1), so the CFT-positive combination is -c_e*Gs_e; plot THAT
# on log.  loc/disp (Gs opposite sign to Gt) -> positive, track +cft_shape.  A wrong-sign Gs (Gs same sign as Gt) gives -c_e*Gs_e < 0 -> it DROPS OFF the log axis -- shown honestly.
dt = np.arange(1, int(Nt_cft/2)); t = dt*at_cft; ref = 8
plt.figure()
for L in [1, 2, 4]:
    for nm, fg, ft, mk in [('loc', loc1_Gs, loc1_Gt, 'o-'), ('disp', disp1_Gs, disp1_Gt, 'd-.'), ('exact', ex_Gs, ex_Gt, 's--')]:
        try:
            gs = fg(L); gt = ft(L); c = cft_shape(t[ref-1]) / gt[ref]   # SAME c as tp
            plt.plot(t, -c*gs[dt], mk, ms=3, label='%s L=%d' % (nm, L))
        except (OSError, KeyError):
            pass
plt.plot(t, cft_shape(t), 'k-', label='(4.28) shape (= $-c\,G_s$ if ratio $-1$)')
plt.yscale('log'); plt.xlabel(r'$t = a_t n_t$'); plt.ylabel(r'$-c\,G_s$ (CFT-positive if ratio $-1$)')
plt.title('sp: $-c\,G_s$ with the tp constant; wrong-sign $G_s$ falls off the log axis'); plt.legend(fontsize=8)
plt.savefig('comp_trio_1_sum_sp.pdf', bbox_inches='tight')

In [ ]:
# SIGN table at dt=5 (raw single-insertion values; no matching).  Same sign => agree.
print('%-6s %-13s %-13s %-13s | %-15s %-13s %-13s' % (
    'L','loc1_Gt','disp1_tp','exact1_tp','loc1_Gs','disp1_sp','exact1_sp'))
for L in [1, 2, 4]:
    def g(f):
        try: return '% .3e' % f(L)[5]
        except (OSError, KeyError): return '   --      '
    print('L=%-4d %-13s %-13s %-13s | %-15s %-13s %-13s' % (
        L, g(loc1_Gt), g(disp1_Gt), g(ex_Gt), g(loc1_Gs), g(disp1_Gs), g(ex_Gs)))
print('\n(tp site 0: loc1_s3 / disp1_tp vs exact1_tp.  sp link 0: loc1 / disp1 vs exact1_sp -- if the')
print(' ultralocal (loc1/disp1) flip vs exact1 that is the genuine GW/resolvent dressing of K.)')

In [ ]:
# RATIO G_s/G_t: the normalization c cancels (same c for tp and sp), leaving the PHYSICAL ratio WITH its
# sign.  loc -> -1 exactly; disp -> -1 from above with L (point-split spatial -> on-site, discretization);
# exact is awkward (the GW sp sign) -- shown honestly, autoscaled, NOT clipped.  CFT target -1.
dt = np.arange(1, int(Nt_cft/2)); t = dt*at_cft; ref = 8
plt.figure()
print('%-6s %-16s %-16s %-16s' % ('L', 'loc Gs/Gt', 'disp Gs/Gt', 'exact Gs/Gt'))
for L in [1, 2, 4]:
    row = []
    for nm, fg, ft, mk in [('loc', loc1_Gs, loc1_Gt, 'o-'), ('disp', disp1_Gs, disp1_Gt, 'd-.'), ('exact', ex_Gs, ex_Gt, '^:')]:
        try:
            r = fg(L) / ft(L); plt.plot(t, r[dt], mk, ms=3, label='%s L=%d' % (nm, L)); row.append('% .4f' % r[ref])
        except (OSError, KeyError):
            row.append('   --   ')
    print('L=%-4d %-16s %-16s %-16s' % (L, *row))
plt.axhline(-1.0, color='k', lw=1, label=r'CFT $-1$')
plt.xlabel(r'$t = a_t n_t$'); plt.ylabel(r'$G_s/G_t$ (raw; c and sign cancel)')
plt.title('summed ratio $G_s/G_t$ (shown honestly, autoscaled)'); plt.legend(fontsize=8)
plt.savefig('comp_trio_1_sum_ratio.pdf', bbox_inches='tight')